# w9_big.ipynb — DC-ASGD 大规模异步模拟(@512 四臂)

User (2026-07-20): 单卡模拟异步参数服务器——M 个轮转 worker 各持一份
陈旧拉取(恒定陈旧度 M−1),worker 梯度在**其拉取时的权重**上计算(同
一模型对象换装权重,现有步代码零改动),PS 用 DC-ASGD 泰勒补偿
g + λ·g⊙g·(W_t − W_pull)(对角 Hessian 近似)后在当前权重上 AdamW
更新,worker 重拉。塔 0.36M → 每份拉取 1.4MB,簿记免费。四臂
@512/2000ep:`_as1`(同路径同步对照)/`_as8`(纯 ASGD,λ=0)/
`_as8dc5`(λ=0.5)/`_as8dc20`(λ=2.0)。参照 = 固定分割 i2ce@512
(.926/.657,tag .721/.745)。判据:as8 掉多少 = 陈旧度的代价;dc 臂
救回多少 = 补偿的价值;as1≈参照 = 代码路径无副作用。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"

ARM = "wcle_i2ce_icetf"
CAP = 512
JOBS = [(["--async-workers", "1"], "_as1", 2000),
        (["--async-workers", "8"], "_as8", 2000),
        (["--async-workers", "8", "--dc-lambda", "0.5"], "_as8dc5", 2000),
        (["--async-workers", "8", "--dc-lambda", "2.0"], "_as8dc20", 2000)]
os.makedirs(OUT_DIR, exist_ok=True)
print("towers:", [f"w9_{ARM}{sfx}@{ep}ep" for _, sfx, ep in JOBS])


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run both spb towers (one per GPU when available).
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
todo = []
for extra, sfx, ep_j in JOBS:
    nm = J.fs_label(ARM, CAP, False, 0, "clean", 16) + sfx
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{ep_j}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((extra, nm, ep_j))
print(f"{len(todo)} tower(s) to run")
stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, extra, nm, ep_j):
    ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} fresh/held -- waiting 130s (corpse window)", flush=True)
        time.sleep(130)
        ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", ARM, "--anchor-cap", str(CAP),
           "--epochs", str(ep_j), "--ckpt-every", str(J.CKPT_EVERY),
           "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
           "--topup-seeds", str(J.TOPUP_SEEDS)] + extra + [
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm} @{ep_j}ep", flush=True)
    t0 = time.time()
    with open(logd / f"{nm}.log", "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
    if p.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

ths = [threading.Thread(target=run_one, args=(gpus[i % len(gpus)], e, nm, ej))
       for i, (e, nm, ej) in enumerate(todo)]
for i, t in enumerate(ths):
    if i:
        time.sleep(120)   # stagger data-load host-RAM transients
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")


In [ ]:
# Readout: async ladder vs the sync i2ce@512 reference.
import json
from pathlib import Path
def show(nm, lab):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        print(f"{lab:28s} ep{d['best_ep']:>4} neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    else:
        print(f"{lab:28s} (pending)")
show("w9_wcle_i2ce_icetf_as1", "as1 sync control")
show("w9_wcle_i2ce_icetf_as8", "as8 plain ASGD (stale 7)")
show("w9_wcle_i2ce_icetf_as8dc5", "as8 + DC lambda 0.5")
show("w9_wcle_i2ce_icetf_as8dc20", "as8 + DC lambda 2.0")
show("w9_wcle_i2ce_icetf", "i2ce@512 sync reference")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
